
# Desarrollo de DAGs

Objetivo:
Crear DataFrame, transformar, escribir/leer Delta Lake, demostrar MERGE (UPSERT) y Time Travel.

## 1. Celda 1: Crear SparkSession con Delta Lake

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("ETL_Delta_Intro")
    .master("local[*]")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "1500m")
    .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate())

print("SparkSession con Delta Lake creada")
print(f"Versión Spark: {spark.version}")

26/06/23 00:00:09 WARN Utils: Your hostname, david-VMware-Virtual-Platform resolves to a loopback address: 127.0.1.1, but we couldn't find any external IP address!
26/06/23 00:00:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/23 00:00:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/06/23 00:00:11 WARN MacAddressUtil: Failed to find a usable hardware address from the network interfaces; using random bytes: 48:05:e1:3c:91:3e:e0:37


SparkSession con Delta Lake creada
Versión Spark: 3.5.8


## 2. Celda 2: Crear Datos de Prueba (CSV simulado)

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

data = [
    ("P001", "Laptop Pro", "Electrónica", 2500.00, 10),
    ("P002", "Mouse Gamer", "Electrónica", 45.50, 50),
    ("P003", "Teclado Mecánico", "Electrónica", 120.00, 30),
    ("P004", "Monitor 4K", "Electrónica", 450.00, 15),
    ("P005", "Silla Ergonómica", "Muebles", 300.00, 8),
]

schema = StructType([
    StructField("id_producto", StringType(), False),
    StructField("nombre", StringType(), False),
    StructField("categoria", StringType(), False),
    StructField("precio", DoubleType(), False),
    StructField("stock", IntegerType(), False),
])

df = spark.createDataFrame(data, schema)
print("DataFrame creado:")
df.show()

DataFrame creado:


+-----------+----------------+-----------+------+-----+
|id_producto|          nombre|  categoria|precio|stock|
+-----------+----------------+-----------+------+-----+
|       P001|      Laptop Pro|Electrónica|2500.0|   10|
|       P002|     Mouse Gamer|Electrónica|  45.5|   50|
|       P003|Teclado Mecánico|Electrónica| 120.0|   30|
|       P004|      Monitor 4K|Electrónica| 450.0|   15|
|       P005|Silla Ergonómica|    Muebles| 300.0|    8|
+-----------+----------------+-----------+------+-----+



## 3. Celda 3: Transformaciones Básicas

In [3]:
from pyspark.sql.functions import col, when, lit, round as spark_round

df_transformado = (df
    .withColumn("precio_iva", spark_round(col("precio") * 1.19, 2))
    .withColumn("estado_stock", when(col("stock") > 20, "Suficiente")
    .when(col("stock") > 10, "Moderado")
    .otherwise("Bajo"))
    .withColumn("fecha_carga", lit("2024-05-20")))

print("Transformaciones aplicadas:")
df_transformado.show()

Transformaciones aplicadas:


+-----------+----------------+-----------+------+-----+----------+------------+-----------+
|id_producto|          nombre|  categoria|precio|stock|precio_iva|estado_stock|fecha_carga|
+-----------+----------------+-----------+------+-----+----------+------------+-----------+
|       P001|      Laptop Pro|Electrónica|2500.0|   10|    2975.0|        Bajo| 2024-05-20|
|       P002|     Mouse Gamer|Electrónica|  45.5|   50|     54.14|  Suficiente| 2024-05-20|
|       P003|Teclado Mecánico|Electrónica| 120.0|   30|     142.8|  Suficiente| 2024-05-20|
|       P004|      Monitor 4K|Electrónica| 450.0|   15|     535.5|    Moderado| 2024-05-20|
|       P005|Silla Ergonómica|    Muebles| 300.0|    8|     357.0|        Bajo| 2024-05-20|
+-----------+----------------+-----------+------+-----+----------+------------+-----------+



## 4. Celda 4: Escribir a Delta Lake

In [4]:
import os
delta_path = os.path.expanduser("~/projects/data/productos_delta")

# Eliminar si existe (para pruebas limpias)
import shutil
if os.path.exists(delta_path):
    shutil.rmtree(delta_path)

df_transformado.write.format("delta").save(delta_path)
print(f"Delta Lake escrito en: {delta_path}")

Delta Lake escrito en: /home/david/projects/data/productos_delta


## 5. Celda 5: Leer desde Delta Lake

In [5]:
df_delta = spark.read.format("delta").load(delta_path)
print("Datos leídos desde Delta Lake:")
df_delta.show()

Datos leídos desde Delta Lake:


26/06/23 00:00:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+----------------+-----------+------+-----+----------+------------+-----------+
|id_producto|          nombre|  categoria|precio|stock|precio_iva|estado_stock|fecha_carga|
+-----------+----------------+-----------+------+-----+----------+------------+-----------+
|       P003|Teclado Mecánico|Electrónica| 120.0|   30|     142.8|  Suficiente| 2024-05-20|
|       P004|      Monitor 4K|Electrónica| 450.0|   15|     535.5|    Moderado| 2024-05-20|
|       P005|Silla Ergonómica|    Muebles| 300.0|    8|     357.0|        Bajo| 2024-05-20|
|       P001|      Laptop Pro|Electrónica|2500.0|   10|    2975.0|        Bajo| 2024-05-20|
|       P002|     Mouse Gamer|Electrónica|  45.5|   50|     54.14|  Suficiente| 2024-05-20|
+-----------+----------------+-----------+------+-----+----------+------------+-----------+



## 6. Celda 6: MERGE (UPSERT) – Actualizar e Insertar

In [6]:
from delta import DeltaTable
# Nuevos datos: P001 actualiza precio, P006 es nuevo
nuevos_datos = [
    ("P001", "Laptop Pro", "Electrónica", 2700.00, 12), # UPDATE
    ("P006", "Webcam HD", "Electrónica", 80.00, 25), # INSERT
]
df_nuevos = spark.createDataFrame(nuevos_datos, schema)
delta_table = DeltaTable.forPath(spark, delta_path)

# MERGE con columnas base
delta_table.alias("target").merge(
    df_nuevos.alias("source"), "target.id_producto = source.id_producto"
).whenMatchedUpdate(set={
    "nombre": "source.nombre",
    "categoria": "source.categoria",
    "precio": "source.precio",
    "stock": "source.stock"
}).whenNotMatchedInsert(values={
    "id_producto": "source.id_producto",
    "nombre": "source.nombre",
    "categoria": "source.categoria",
    "precio": "source.precio",
    "stock": "source.stock"
}).execute()

print("MERGE base ejecutado")

# RECALCULAR columnas derivadas en TODOS los registros
df_actualizado = spark.read.format("delta").load(delta_path)
df_recalculado = df_actualizado \
    .withColumn("precio_iva", spark_round(col("precio") * 1.19, 2)) \
    .withColumn("estado_stock", when(col("stock") > 20, "Suficiente")
    .when(col("stock") > 10, "Moderado")
    .otherwise("Bajo"))

# Sobrescribir la tabla Delta con valores recalculados
df_recalculado.write.format("delta").mode("overwrite").option("overwriteSchema",
"true").save(delta_path)
print("Columnas derivadas recalculadas")
df_recalculado.show()

MERGE base ejecutado


Columnas derivadas recalculadas


+-----------+----------------+-----------+------+-----+----------+------------+-----------+
|id_producto|          nombre|  categoria|precio|stock|precio_iva|estado_stock|fecha_carga|
+-----------+----------------+-----------+------+-----+----------+------------+-----------+
|       P001|      Laptop Pro|Electrónica|2700.0|   12|    3213.0|    Moderado| 2024-05-20|
|       P002|     Mouse Gamer|Electrónica|  45.5|   50|     54.14|  Suficiente| 2024-05-20|
|       P006|       Webcam HD|Electrónica|  80.0|   25|      95.2|  Suficiente|       NULL|
|       P003|Teclado Mecánico|Electrónica| 120.0|   30|     142.8|  Suficiente| 2024-05-20|
|       P004|      Monitor 4K|Electrónica| 450.0|   15|     535.5|    Moderado| 2024-05-20|
|       P005|Silla Ergonómica|    Muebles| 300.0|    8|     357.0|        Bajo| 2024-05-20|
+-----------+----------------+-----------+------+-----+----------+------------+-----------+



## 7. Celda 7: Verificar Resultado del MERGE

In [7]:
df_resultado = spark.read.format("delta").load(delta_path)
print("Datos después del MERGE:")
df_resultado.orderBy("id_producto").show()

Datos después del MERGE:


+-----------+----------------+-----------+------+-----+----------+------------+-----------+
|id_producto|          nombre|  categoria|precio|stock|precio_iva|estado_stock|fecha_carga|
+-----------+----------------+-----------+------+-----+----------+------------+-----------+
|       P001|      Laptop Pro|Electrónica|2700.0|   12|    3213.0|    Moderado| 2024-05-20|
|       P002|     Mouse Gamer|Electrónica|  45.5|   50|     54.14|  Suficiente| 2024-05-20|
|       P003|Teclado Mecánico|Electrónica| 120.0|   30|     142.8|  Suficiente| 2024-05-20|
|       P004|      Monitor 4K|Electrónica| 450.0|   15|     535.5|    Moderado| 2024-05-20|
|       P005|Silla Ergonómica|    Muebles| 300.0|    8|     357.0|        Bajo| 2024-05-20|
|       P006|       Webcam HD|Electrónica|  80.0|   25|      95.2|  Suficiente|       NULL|
+-----------+----------------+-----------+------+-----+----------+------------+-----------+



## 8. Celda 8: Time Travel – Ver Historial

In [8]:
delta_table = DeltaTable.forPath(spark, delta_path)
print("Historial de versiones:")
delta_table.history().select("version", "timestamp",
"operation").show(truncate=False) 

Historial de versiones:


+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|2      |2026-06-23 00:01:01.658|WRITE    |
|1      |2026-06-23 00:00:52.678|MERGE    |
|0      |2026-06-23 00:00:27.289|WRITE    |
+-------+-----------------------+---------+



## 9. Celda 9: Time Travel – Leer Versión Anterior

In [9]:
df_version_0 = spark.read.format("delta").option("versionAsOf",0).df_version_0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)
print("Versión 0 (original, sin P006, P001=2500):")

df_version_0.select("id_producto", "nombre", "precio").show()
df_version_1 = spark.read.format("delta").option("versionAsOf",1).load(delta_path)
print("Versión 1 (después de MERGE, con P006, P001=2700):")

df_version_1.select("id_producto", "nombre", "precio").show()
print("Versión 0 (original, sin P006, P001=2500):")

df_version_0.select("id_producto", "nombre", "precio").show()

Versión 0 (original, sin P006, P001=2500):


+-----------+----------------+------+
|id_producto|          nombre|precio|
+-----------+----------------+------+
|       P003|Teclado Mecánico| 120.0|
|       P004|      Monitor 4K| 450.0|
|       P005|Silla Ergonómica| 300.0|
|       P001|      Laptop Pro|2500.0|
|       P002|     Mouse Gamer|  45.5|
+-----------+----------------+------+

Versión 1 (después de MERGE, con P006, P001=2700):


+-----------+----------------+------+
|id_producto|          nombre|precio|
+-----------+----------------+------+
|       P003|Teclado Mecánico| 120.0|
|       P004|      Monitor 4K| 450.0|
|       P005|Silla Ergonómica| 300.0|
|       P001|      Laptop Pro|2700.0|
|       P002|     Mouse Gamer|  45.5|
|       P006|       Webcam HD|  80.0|
+-----------+----------------+------+

Versión 0 (original, sin P006, P001=2500):


+-----------+----------------+------+
|id_producto|          nombre|precio|
+-----------+----------------+------+
|       P003|Teclado Mecánico| 120.0|
|       P004|      Monitor 4K| 450.0|
|       P005|Silla Ergonómica| 300.0|
|       P001|      Laptop Pro|2500.0|
|       P002|     Mouse Gamer|  45.5|
+-----------+----------------+------+

